# The Anatomy of AI · Lecture 4 · Reading a Small Network
### The notebook

Lecture 3's network we built ourselves, so we knew what each neuron was for. Here you train a network you did *not* build,
then read it the way the whole field reads networks: **look** at each neuron, **guess** its job, and **test** the guess by
switching the neuron off. Then you train 200 more and ask whether the same jobs come back in the same neurons.

Each section names the lecture plate it rebuilds. A `check(...)` line prints ✓ when your number matches the lecture's,
and stops the notebook if it does not. Five **Your turn** exercises sit along the way; solutions are at the end.

It runs on the CPU in a few minutes; one cell trains 200 networks and takes most of that. No GPU, no downloads, no keys.

In [ ]:
# Pinned to the versions free Colab ships (Python 3.12); nothing else is needed, and no keys.
%pip install -q numpy==2.0.2 matplotlib==3.10.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "−")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value) + 0.0:.6g}"          # + 0.0 prints −0 as 0
    else:
        ok = value == expected
        shown = repr(value)
    want = f"{expected:.6g}" if isinstance(expected, float) else expected
    print(f"{'✓' if ok else '✗'} {name}: {shown}   (lecture: {want})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## Plate II · A richer diary

A longer diary, and fussier tastes: 300 days, still just warmth and rain. You enjoy a walk only when it is

1. not too cold (warmth above 0.25),
2. not too hot (warmth below 0.80),
3. not too wet (rain below 0.60),
4. and the cooler it is, the less rain you'll put up with (rain below warmth: not *cold and damp*).

Those are the four **walls** of the enjoyed region. We made the rule up, so we hold the answer key. Real researchers never
do; here it lets us check every name we give a neuron. The lecture drew the days with Python's random generator, seed 4,
leaving out days within 0.03 of a wall.

In [ ]:
import math, random
from collections import Counter

# each wall as a·(warmth, rain) + c = 0, positive on the side that is NOT enjoyed
WALLS = {"too cold": (-1.0, 0.0, 0.25), "too hot": (1.0, 0.0, -0.80), "too wet": (0.0, 1.0, -0.60), "cold and damp": (-1.0, 1.0, 0.0)}
# the piece of each wall that is an edge of the enjoyed region
EDGES = {"too cold": ((0.25, 0.0), (0.25, 0.25)), "too hot": ((0.80, 0.0), (0.80, 0.60)),
         "too wet": ((0.60, 0.60), (0.80, 0.60)), "cold and damp": ((0.25, 0.25), (0.60, 0.60))}

def enjoyed(w, r):
    return all(a1 * w + a2 * r + c < 0 for a1, a2, c in WALLS.values())

def near_an_edge(w, r, gap):
    def seg_dist(p, a, b):
        (ax, ay), (bx, by), (px, py) = a, b, p
        dx, dy = bx - ax, by - ay
        t = max(0.0, min(1.0, ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)))
        return math.hypot(px - ax - t * dx, py - ay - t * dy)
    return min(seg_dist((w, r), a, b) for a, b in EDGES.values()) < gap

def rich_diary(n=300, seed=4, gap=0.03):
    rng, days = random.Random(seed), []
    while len(days) < n:
        w, r = rng.uniform(0.02, 0.98), rng.uniform(0.02, 0.98)
        if near_an_edge(w, r, gap):
            continue
        days.append((w, r, enjoyed(w, r)))
    return days

diary = rich_diary()
X = np.array([(w, r) for w, r, _ in diary])
Y = np.array([1.0 if e else 0.0 for *_, e in diary])
check("days", len(X), 300)
check("enjoyed", int(Y.sum()), 69)
check("the first day, as the lecture drew it", (round(float(X[0, 0]), 4), round(float(X[0, 1]), 4), bool(Y[0])), (0.4002, 0.1688, True))

# a dense map of every day, 101 × 101, for reading what a network does everywhere
G = np.array([(w, r) for r in np.linspace(0, 1, 101) for w in np.linspace(0, 1, 101)])

In [ ]:
def draw_diary(ax, walls=True, dot=3):
    for (w, r), y in zip(X, Y):
        ax.plot(w, r, "o", ms=dot, mfc=INK if y else PAPER, mec=INK, mew=0.7)
    if walls:
        for (p, q) in EDGES.values():
            ax.plot(*zip(p, q), color=INK, lw=2, ls="--")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")

fig, ax = plt.subplots(figsize=(4.6, 4.6))
draw_diary(ax, dot=4)
for name, (x, y) in {"too cold": (0.12, 0.12), "too hot": (0.9, 0.3), "too wet": (0.7, 0.66), "cold and damp": (0.25, 0.47)}.items():
    ax.text(x, y, name, ha="center", style="italic", fontsize=9, bbox=dict(fc=PAPER, ec="none", pad=1))
ax.set_xlabel("warmth"); ax.set_ylabel("rain"); ax.set_title("300 days, 69 enjoyed; the four walls dashed", fontsize=10)
plt.show()

## Plate III · The specimen

The network is Lecture 3's, with eight hidden neurons: 2 inputs → 8 hidden neurons → 1 combiner, every neuron gated. It
starts the way Lecture 3's did (random weights by He's rule, every hidden line through the middle of the diary, the
combiner's bias at 0.5) and learns for 30,000 steps of size 0.1.

We train **200** of them side by side, one per random start (*seed*) 0 to 199, because later plates need them all. The
specimen is seed 0. `Nets` holds them; it is Lecture 3's code written with matrix products, so it runs fast. A weight
decay `wd` (Plate XII) pulls every weight a little towards 0 at each step.

In [ ]:
class Nets:
    """Many copies of a 2 → H → 1 network (one per seed): hidden weights A (S×H×2), biases c, votes v, combiner bias d."""
    def __init__(self, seeds, H, wd=0.0):
        A, v = [], []
        for s in seeds:
            rng = np.random.default_rng(s)
            A.append(rng.normal(0, np.sqrt(2 / 2), (H, 2)))
            v.append(rng.normal(0, np.sqrt(2 / H), H))
        self.A, self.v = np.array(A), np.array(v)
        self.c = -self.A @ np.array([0.5, 0.5])          # every hidden line through the middle of the diary
        self.d = np.full(len(self.A), 0.5)                # the combiner fires on every day at first
        self.wd = wd

    def forward(self, P, off=None):
        """Run days P through every network. off: a 0/1 mask (S×H or H) that switches hidden neurons off."""
        z = P @ self.A.transpose(0, 2, 1) + self.c[:, None, :]       # S × days × H
        h = np.maximum(0.0, z)
        if off is not None:
            h = h * np.asarray(off).reshape(-1, 1, h.shape[2]) if np.ndim(off) == 2 else h * off
        t = (h @ self.v[:, :, None])[..., 0] + self.d[:, None]
        return z, h, t, np.maximum(0.0, t)

    def train(self, steps=30000, lr=0.1):
        n = len(X)
        for _ in range(steps):
            z, h, t, out = self.forward(X)
            b_t = 2 * (out - Y) / n * (t > 0)                         # blame on each answer, through the combiner's gate
            g_v = (h.transpose(0, 2, 1) @ b_t[:, :, None])[..., 0]
            b_z = b_t[:, :, None] * self.v[:, None, :] * (z > 0)     # back along each vote, through each gate
            g_A = b_z.transpose(0, 2, 1) @ X
            if self.wd:
                g_A, g_v = g_A + self.wd * self.A, g_v + self.wd * self.v
            self.A -= lr * g_A; self.c -= lr * b_z.sum(1); self.v -= lr * g_v; self.d -= lr * b_t.sum(1)
        return self

    def right(self, P=X, T=Y, off=None):
        return ((self.forward(P, off)[3] > 0.5) == (T > 0.5)).sum(axis=1)

    def one(self, s):
        """Network s alone (a view, not a copy), so a big map of days runs through one network only."""
        m = object.__new__(Nets)
        m.A, m.c, m.v, m.d, m.wd = self.A[s:s + 1], self.c[s:s + 1], self.v[s:s + 1], self.d[s:s + 1], self.wd
        return m

In [ ]:
import time
t0 = time.time()
nets = Nets(range(200), 8).train()           # the slow cell: 200 networks, 30,000 steps each
print(f"trained 200 networks in {time.time() - t0:.0f} s")
right = nets.right()
check("the specimen (seed 0): days right", int(right[0]), 300)
check("networks that get all 300 right", float(np.mean(right == 300)), 0.81, tol=0.02)
print("the specimen's 33 numbers:", 8 * 2 + 8 + 8 + 1)

In [ ]:
def net(s):
    """One network's numbers, as a small dict."""
    return dict(A=nets.A[s], c=nets.c[s], v=nets.v[s], d=nets.d[s])

def verdict(ax, s, off=None, M=nets, hatch="////"):
    g = np.linspace(0, 1, 151); Wg, Rg = np.meshgrid(g, g)
    out = M.one(s).forward(np.c_[Wg.ravel(), Rg.ravel()], off)[3][0].reshape(Wg.shape)
    if (out > 0.5).any():
        ax.contourf(Wg, Rg, out, levels=[0.5, max(out.max(), 0.51)], colors="none", hatches=[hatch])

def draw_network(ax, s, M=nets, off=()):
    """2 → H → 1, wires as thick as their weight, hollow if negative; switched-off neurons crossed out."""
    H = M.A.shape[1]
    IN, OUT = [(0, 0.7), (0, 0.3)], (2, 0.5)
    HID = [(1, 1 - (j + 0.5) / H) for j in range(H)]
    def wire(p, q, w, faint):
        lw = 0.4 + 0.9 * min(abs(w), 4)
        ax.plot(*zip(p, q), color=INK2 if faint else INK, lw=lw, solid_capstyle="butt", alpha=0.35 if faint else 1)
        if w < 0 and lw > 1.6 and not faint:
            ax.plot(*zip(p, q), color=PAPER, lw=lw - 1.2, solid_capstyle="butt")
    for j, p in enumerate(HID):
        for i in range(2):
            wire(IN[i], p, M.A[s, j, i], j in off)
        wire(p, OUT, M.v[s, j], j in off)
    for p in IN + HID + [OUT]:
        ax.add_patch(plt.Circle(p, 0.045, fc=PAPER, ec=INK, zorder=3))
    for j, p in enumerate(HID):
        ax.text(*p, str(j), ha="center", va="center", fontsize=8, zorder=4)
        if j in off:
            ax.plot([p[0] - .07, p[0] + .07], [p[1] - .07, p[1] + .07], color=INK, lw=2, zorder=5)
            ax.plot([p[0] - .07, p[0] + .07], [p[1] + .07, p[1] - .07], color=INK, lw=2, zorder=5)
    ax.text(-0.08, 0.7, "warmth", ha="right", va="center", fontsize=9); ax.text(-0.08, 0.3, "rain", ha="right", va="center", fontsize=9)
    ax.text(2, 0.38, "combiner", ha="center", va="top", fontsize=9)
    ax.set_xlim(-0.45, 2.2); ax.set_ylim(-0.05, 1.05); ax.set_aspect("equal"); ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
draw_network(axes[0], 0)
verdict(axes[1], 0); draw_diary(axes[1]); axes[1].set_title("its verdict, hatched where it says go: 300 of 300", fontsize=10)
plt.show()

## Plate IV · Reading one neuron

A neuron's whole story fits on one page:

- **its line**, where its total is 0, and the side it fires on;
- **how many days** it fires on;
- **its vote**: its weight into the combiner, for (positive) or against (negative), and how strongly.

We also ask whether its line **sits on a wall**: every point of the wall's edge within 0.06 of the line.

In [ ]:
def line_distance(a, c, p0, p1, n=21):
    """The largest distance from points along a wall's edge to the line a·x + c = 0."""
    t = np.linspace(0, 1, n)[:, None]
    pts = np.array(p0) * (1 - t) + np.array(p1) * t
    return float(np.max(np.abs(pts @ a + c) / (np.linalg.norm(a) + 1e-12)))

def read(s):
    """Read every hidden neuron of network s by its line: days fired on, vote, nearest wall, and which side of it fires."""
    z = X @ nets.A[s].T + nets.c[s]
    rows = []
    for j in range(nets.A.shape[1]):
        a, c, v = nets.A[s, j], nets.c[s, j], nets.v[s, j]
        fires = int(np.sum(z[:, j] > 0))
        if fires == 0:
            rows.append(dict(neuron=j, fires_on=0, vote=v, on_wall=None, nearest=None, dist=None, side=None)); continue
        nearest, dist = min(((name, line_distance(a, c, *EDGES[name])) for name in EDGES), key=lambda q: q[1])
        side = "beyond the wall" if a @ np.array(WALLS[nearest][:2]) > 0 else "the enjoyed side"
        rows.append(dict(neuron=j, fires_on=fires, vote=v, on_wall=nearest if dist < 0.06 else None, nearest=nearest, dist=dist, side=side))
    return rows

page = read(0)
n0 = page[0]
print(f"neuron 0 fires on {n0['fires_on']} days, votes {n0['vote']:+.2f}; its line is nearest the {n0['nearest']} wall, {n0['dist']:.4f} away")
check("neuron 0 fires on", n0["fires_on"], 117)
check("neuron 0 votes", round(n0["vote"], 2), -2.53)
check("neuron 0's line sits on a wall", n0["on_wall"], None)

def neuron_page(ax, s, j):
    a, c = nets.A[s, j], nets.c[s, j]
    g = np.linspace(0, 1, 151); Wg, Rg = np.meshgrid(g, g)
    Z = a[0] * Wg + a[1] * Rg + c
    if Z.max() > 0:
        ax.contourf(Wg, Rg, Z, levels=[0, Z.max()], colors="none", hatches=["\\\\\\"])
    if Z.min() < 0 < Z.max():
        ax.contour(Wg, Rg, Z, levels=[0], colors=INK, linewidths=2)
    draw_diary(ax, dot=2)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
neuron_page(ax, 0, 0)
ax.set_title(f"neuron 0: fires on 117 days (hatched), votes {n0['vote']:+.2f}", fontsize=10)
plt.show()

## Plate V · The catalogue

Do it for all eight. Not one line sits on a wall. Two neurons have no line in the diary at all: they fire on every day.
Reading lines alone, we can't name them.

Rumelhart, Hinton and Williams (1986) read their family-trees network unit by unit: one unit for nationality, one for
generation, one for branch of the family. The lecture's margin note compares: across 200 of our networks a *textbook*
hot alarm (a neuron whose line sits on the hot wall, fires beyond it, and votes against) appears in 113.

In [ ]:
for row in page:
    print(f"neuron {row['neuron']}: fires on {row['fires_on']:3d} days, votes {row['vote']:+.2f}, "
          f"nearest the {row['nearest']} wall, {row['dist']:.3f} away")
check("days each neuron fires on", [r["fires_on"] for r in page], [117, 243, 70, 17, 300, 2, 211, 300])
check("lines that sit on a wall", sum(r["on_wall"] is not None for r in page), 0)
check("neuron 6's line, from the hot wall", round(page[6]["dist"], 3), 0.076)

fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))
for j, ax in enumerate(axes.ravel()):
    neuron_page(ax, 0, j)
    ax.set_title(f"neuron {j}: fires on {page[j]['fires_on']}, votes {page[j]['vote']:+.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

textbook = []
for s in range(200):
    for r in read(s):
        if r["on_wall"] == "too hot" and r["side"] == "beyond the wall" and r["vote"] < 0:
            textbook.append((s, r["neuron"]))
with_one = len({s for s, _ in textbook})
print(f"networks with a textbook hot alarm: {with_one} of 200; such neurons by slot 0–7:",
      [sum(1 for _, j in textbook if j == k) for k in range(8)])
check("networks with a textbook hot alarm", with_one, 113, tol=3)

## Plate VI · Switch one off

Now test. **Switching a neuron off** means it says 0 whatever the day (the field calls this *ablation*). Silence one
neuron and see which days change their verdict, and where they lie.

- Neuron 0 off: 32 hot days come through. Whatever its line looks like, its job is the hot wall.
- Neuron 4 off: every enjoyed day is rejected. It votes *for* walks.
- Neuron 6 off: its line lies near the hot wall, yet the cold, damp days come through. A line is where a neuron starts to
  fire, not where it changes the verdict.
- Neuron 2 off: nothing changes at all.

In [ ]:
def region(w, r):
    beyond = [n for n, (a1, a2, c) in WALLS.items() if a1 * w + a2 * r + c > 0]
    return "+".join(beyond) if beyond else "inside"

def switch_off(s, neurons, P=X, T=Y):
    """Switch some neurons of network s off: which days change, and where they lie."""
    mask = np.ones(nets.A.shape[1]); mask[list(neurons)] = 0
    one = nets.one(s)
    before = one.forward(P)[3][0] > 0.5
    after = one.forward(P, mask)[3][0] > 0.5
    changed = np.where(before != after)[0]
    return changed, Counter(region(*P[i]) for i in changed), int(np.sum(after == (T > 0.5)))

for j, expected in ((0, 34), (4, 69), (6, 135), (2, 0)):
    changed, where, now_right = switch_off(0, [j])
    print(f"neuron {j} off: {len(changed)} days change {dict(where)}; now {now_right} of 300 right")
    check(f"neuron {j} off: days that change", len(changed), expected)
check("neuron 0 off: hot days that get through", switch_off(0, [0])[1]["too hot"], 32)
check("neuron 4 off: enjoyed days rejected", switch_off(0, [4])[1]["inside"], 69)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, j in zip(axes, (0, 4, 6, 2)):
    mask = np.ones(8); mask[j] = 0
    verdict(ax, 0, mask)
    changed, _, _ = switch_off(0, [j])
    draw_diary(ax, dot=2)
    ax.plot(X[changed, 0], X[changed, 1], "o", ms=8, mfc="none", mec=INK, mew=0.9)
    ax.set_title(f"neuron {j} off: {len(changed)} days change (ringed)", fontsize=10)
plt.tight_layout(); plt.show()

## Plate VII · What switching off reveals

Name every neuron by what breaks when it is switched off, using the dense map of 10,201 days:

- **go** if switching it off rejects enjoyed days (it was holding them in);
- a **wall** if days beyond that wall now get through (it is that wall's alarm);
- **idle** if under 0.5% of the map changes; **dead** if it is silent on every diary day;
- **two jobs** if its second-biggest share of the change is at least 30%.

A day beyond two walls counts half for each.

In [ ]:
JOBS = ["go", "too cold", "too hot", "too wet", "cold and damp"]
SHARE = np.zeros((len(G), len(JOBS)))                     # each map day's share of each job
for i, (w, r) in enumerate(G):
    reg = region(w, r)
    if reg == "inside":
        SHARE[i, 0] = 1
    else:
        parts = reg.split("+")
        for p in parts:
            SHARE[i, JOBS.index(p)] += 1 / len(parts)

def jobs_of(s, M=nets):
    H, one = M.A.shape[1], M.one(s)
    base = one.forward(G)[3][0] > 0.5
    zx = X @ M.A[s].T + M.c[s]
    rows = []
    for j in range(H):
        if not np.any(zx[:, j] > 0):
            rows.append(("dead", None)); continue
        mask = np.ones(H); mask[j] = 0
        flip = (one.forward(G, mask)[3][0] > 0.5) != base
        if flip.mean() < 0.005:
            rows.append(("idle", None)); continue
        share = SHARE[flip].sum(0)
        order = np.argsort(-share, kind="stable")
        second = JOBS[order[1]] if share[order[1]] / share.sum() >= 0.3 else None
        rows.append((JOBS[order[0]], second))
    return rows

t0 = time.time()
jobs = [jobs_of(s) for s in range(200)]
print(f"named 1,600 neurons in {time.time() - t0:.0f} s")
spec = [j for j, _ in jobs[0]]
print("the specimen:", spec)
check("the specimen's jobs, neuron by neuron", spec, ["too hot", "too hot", "idle", "idle", "go", "idle", "cold and damp", "go"])

count = Counter(j for rows in jobs for j, _ in rows)
working = [(j, k) for rows in jobs for j, k in rows if j not in ("idle", "dead")]
two = sum(1 for _, k in working if k)
print("1,600 neurons by job:", dict(count.most_common()))
for name, expected in (("too hot", 393), ("go", 337), ("idle", 325), ("cold and damp", 249), ("too wet", 132), ("too cold", 124), ("dead", 40)):
    check(f"neurons whose job is {name}", count[name], expected, tol=8)
check("working neurons with two jobs", two / len(working), 310 / 1235, tol=0.01)

## Plate VIII · Teams

Some walls are built by a team. Random start 2 has two neurons on the hot wall: switch off neuron 2 and 26 days change;
switch off neuron 5 and not one; switch off both and 48 days change, 22 more than neuron 2 alone. Switching off one
neuron at a time can make a neuron look unimportant when it isn't. The specimen's three idle neurons, though, really are
idle: all off together, nothing changes.

In [ ]:
for neurons, expected in (([2], 26), ([5], 0), ([2, 5], 48)):
    check(f"random start 2, neurons {neurons} off: days that change", len(switch_off(2, neurons)[0]), expected)
check("the specimen's idle neurons 2, 3 and 5, all off", len(switch_off(0, [2, 3, 5])[0]), 0)

(Plate IX is the narrated film of this lecture.)

## Plate X · Train again

Same diary, same training, a different random start. Start 6 builds four alarms, one per wall, and four idle neurons:
Lecture 3's two-alarm design, grown up. Start 9 builds the very same design, but too cold is neuron 1 in one and neuron 4
in the other; cold and damp is 6 in one and 5 in the other. The jobs are real; which neuron holds them is an accident of
the start.

In [ ]:
for s in (0, 6, 9):
    print(f"start {s}:", [j if k is None else f"{j} / {k}" for j, k in jobs[s]])
check("start 6", [j for j, _ in jobs[6]], ["idle", "too cold", "too hot", "idle", "idle", "idle", "cold and damp", "too wet"])
check("start 9", [j for j, _ in jobs[9]], ["idle", "idle", "too hot", "idle", "too cold", "cold and damp", "too wet", "idle"])

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, s in zip(axes, (0, 6, 9)):
    verdict(ax, s); draw_diary(ax, dot=2)
    ax.set_title(f"start {s}: " + ", ".join(f"{j}:{n}" for j, (n, _) in enumerate(jobs[s]) if n not in ("idle", "dead")), fontsize=8)
plt.tight_layout(); plt.show()

## Plate XI · Do the same jobs come back?

Over 200 starts: which networks have a neuron whose main job is each one, and in which of the eight slots those neurons
sit. The big jobs come back almost every time; the short walls get a neuron of their own about half the time (otherwise a
neighbour takes them as a second job). And no job has a favourite slot.

In [ ]:
present = {jb: sum(any(j == jb for j, _ in rows) for rows in jobs) for jb in JOBS}
slots = {jb: [sum(1 for rows in jobs if rows[k][0] == jb) for k in range(8)] for jb in JOBS}
all_five = sum(all(any(j == jb for j, _ in rows) for jb in JOBS) for rows in jobs)
for jb in ("too hot", "cold and damp", "go", "too wet", "too cold"):
    print(f"{jb:>14}: in {present[jb]:3d} networks; by slot {slots[jb]}")
print("all five jobs in one network:", all_five)
for jb, expected in (("too hot", 194), ("cold and damp", 185), ("go", 180), ("too wet", 117), ("too cold", 110)):
    check(f"networks with a '{jb}' neuron", present[jb], expected, tol=3)
check("networks with all five jobs", all_five, 36, tol=3)
check("the hot alarm's busiest slot holds", max(slots["too hot"]), 54, tol=3)

fig, ax = plt.subplots(figsize=(7.5, 3))
im = ax.imshow([slots[jb] for jb in JOBS], cmap=INKMAP, aspect="auto")
ax.set_yticks(range(5)); ax.set_yticklabels(JOBS); ax.set_xticks(range(8)); ax.set_xlabel("slot (neuron number)")
for i, jb in enumerate(JOBS):
    for k, n in enumerate(slots[jb]):
        ax.text(k, i, n, ha="center", va="center", fontsize=8, color=PAPER if n > 35 else INK)
ax.set_title("how many neurons hold each job, slot by slot: no favourite slot", fontsize=10)
plt.show()

## Plate XII · How many neurons?

Four walls, so four neurons should do. They can, but from a random start only about one network in four gets every day
right. With spare neurons, learning succeeds. We train 100 four-neuron networks to see.

Then **weight decay**: a pull of every weight towards 0 at each step. Rumelhart, Hinton and Williams (1986) added a little
so their weights would be easier to read. On our diary it prunes instead: at 0.001 the spare neurons shrink to almost
nothing, and at 0.01 the specimen drops to 291 of 300.

In [ ]:
t0 = time.time()
four = Nets(range(100), 4).train()
r4 = four.right()
print(f"trained 100 four-neuron networks in {time.time() - t0:.0f} s: all 300 right in {int(np.sum(r4 == 300))}, on average {r4.mean():.1f}")
check("four neurons: all 300 right (of 100)", int(np.sum(r4 == 300)), 23, tol=3)
check("four neurons: days right on average", float(r4.mean()), 294.8, tol=0.5)

for wd, expected in ((0.001, 300), (0.01, 291)):
    m = Nets([0], 8, wd=wd).train()
    size = np.linalg.norm(m.A[0], axis=1)
    print(f"weight decay {wd}: {int(m.right()[0])} of 300 right; each neuron's weight size {np.round(size, 2)}")
    check(f"the specimen with weight decay {wd}: days right", int(m.right()[0]), expected)
    if wd == 0.001:
        check("neurons pruned to almost nothing (weights under 0.1)", int(np.sum(size < 0.1)), 3)

## Plates XIII–XIV · History, and check yourself

Reading hidden units is as old as backpropagation (Rumelhart, Hinton & Williams, 1986). Li, Yosinski and colleagues (2016)
found some features learned one to one across networks and others only as shared groups in different neurons. Morcos and
colleagues and Zhou and colleagues (2018) switched off single units: how selective a unit is says little about how much it
matters overall.

### Your turn 1 · Useless, or in a team?

Random start 1 has two neurons on the wet side, 0 and 5. Switch each off alone, then both together. How many days change?

*Hint:* `switch_off(1, [0])`; the first thing it returns lists the days that change.

In [ ]:
team_of_two = None   # TODO: days that change with neuron 0 off, neuron 5 off, and both off

### Your turn 2 · Same job, same number?

Two networks trained on the same diary both have a hot alarm. Is it the same neuron number? Count, over the 200 networks,
how many hold a too-hot neuron in slot 0.

*Hint:* `jobs[s][0][0]` is network `s`'s neuron 0's main job.

In [ ]:
hot_in_slot_0 = None   # TODO

### Your turn 3 · Read neuron 1 by switching it off

The specimen's neuron 1 fires on 243 days and votes against. Switch it off: how many days change, and beyond which walls?

*Hint:* `switch_off(0, [1])`.

In [ ]:
neuron1_changed = None   # TODO

### Your turn 4 · A line on a wall, but not an alarm

Find a neuron, in any of the 200 networks, whose line sits on the hot wall but which fires on the *enjoyed* side and still
votes against (like the specimen's neuron 6, which comes close). How many such neurons are there?

*Hint:* use `read(s)` and look at `on_wall`, `side` and `vote`.

In [ ]:
odd_ones = None   # TODO

### Your turn 5 · Sixteen neurons

With sixteen neurons, how many of the first 20 starts get all 300 days right? (The lecture's sweep: 98 of 100.)

*Hint:* `Nets(range(20), 16).train()` takes about a minute.

In [ ]:
wins_sixteen = None   # TODO

## What reading a network takes

- Look at each neuron, its line, side and vote, and guess its job.
- Test the guess: switch it off and see what breaks.
- Expect teams, idle neurons and dead ones.
- Jobs are real; which neuron holds them is an accident of the start.

**Next · Part II · Lecture 5, Pictures as Numbers:** two inputs let us draw every neuron's line. A photograph has
thousands. How do we read a neuron then?

---

## Solutions

**Your turn 1.** Alone, 0 and 2 days; together, 11. Neither alone tells the truth about the pair.

In [ ]:
team_of_two = [len(switch_off(1, n)[0]) for n in ([0], [5], [0, 5])]
print("days that change:", team_of_two)
check("random start 1: neuron 0 off, neuron 5 off, both off", team_of_two, [0, 2, 11])

**Your turn 2.** Only a small share: the hot alarm lands in every slot about equally often.

In [ ]:
hot_in_slot_0 = sum(1 for rows in jobs if rows[0][0] == "too hot")
print(f"{hot_in_slot_0} of 200 networks have a too-hot neuron in slot 0")
check("too-hot neurons in slot 0", hot_in_slot_0, 48, tol=3)

**Your turn 3.** 110 days: hot ones, wet ones and cold-and-damp ones. Neuron 1 is a broad alarm on the warm, wet side.

In [ ]:
neuron1_changed, where, _ = switch_off(0, [1])
print(len(neuron1_changed), "days change:", dict(where))
check("neuron 1 off: days that change", len(neuron1_changed), 110)

**Your turn 4.** The lecture counts 41 such neurons on the hot wall (this notebook's faster training may land one off). A line on a wall does not even guarantee that wall is its job.

In [ ]:
odd_ones = sum(1 for s in range(200) for r in read(s) if r["on_wall"] == "too hot" and r["side"] == "the enjoyed side" and r["vote"] < 0)
print("neurons on the hot wall that fire on the enjoyed side and vote against:", odd_ones)
check("such neurons", odd_ones, 41, tol=3)

**Your turn 5.** Nearly all of them: spare neurons make learning succeed.

In [ ]:
wins_sixteen = int(np.sum(Nets(range(20), 16).train().right() == 300))
print(f"{wins_sixteen} of 20")
check("sixteen neurons: nearly every start gets all 300", wins_sixteen >= 18, True)